# Referência de persistência (naïve simples)

**TCC II: Previsão da Demanda Turística em Foz do Iguaçu**

A persistência prevê cada mês repetindo o valor real do mês anterior. É a regra simples que usa
a mesma informação que o Random Forest e a LSTM recebem como defasagem 1 na previsão de um passo
à frente; o naïve sazonal, ao contrário, só usa o valor de 12 meses antes.

**O que este notebook faz e o que não faz**

| Item | Definição |
|---|---|
| Base | abre a base real **somente para leitura**: não treina modelo e não grava execução |
| Teste | os mesmos 12 meses de 2025 das 11 execuções gravadas (RN04, RN07) |
| Previsão | mês t = valor real do mês t-1 (janeiro de 2025 = dezembro de 2024) |
| Métricas | MAE, RMSE e MAPE pela mesma função do sistema (`AvaliacaoService.avaliar`, RF16) |
| Significância | Diebold-Mariano com a correção de Harvey, Leybourne e Newbold, pela mesma função do sistema (RF17, RN08) |
| Saída | `docs/resultados/persistencia.json` |

O naïve sazonal continua no experimento como estava: a persistência é uma segunda referência,
não uma substituta.

In [1]:
# Bibliotecas do Python
import json, os, sys
from pathlib import Path

# Pasta do projeto, para o notebook usar os módulos do sistema
RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ))

# Bibliotecas de cálculo, tabelas e banco de dados
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

# Módulos do sistema
import config
from aplicacao.avaliacao import AvaliacaoService
from dominio.entidades import (EscopoAvaliacaoEnum as E, Previsao, TipoMetricaEnum as M,
                               TipoModeloEnum as T)
from infraestrutura.repositorios.sqlite_repo import SqliteRepository

# Pasta onde o resultado é gravado (docs/resultados, criada se ainda não existir) e banco lido
# (a base real). As variáveis TCC_SAIDA e TCC_BANCO, quando definidas, apontam para outra pasta
# e para uma cópia do banco, para conferir sem tocar nos arquivos oficiais.
SAIDA = Path(os.environ.get("TCC_SAIDA") or RAIZ / "docs" / "resultados")
SAIDA.mkdir(parents=True, exist_ok=True)
BANCO = Path(os.environ.get("TCC_BANCO") or config.BANCO)

# Cria o repositório do banco.
repo = SqliteRepository(BANCO)
# Conexão somente leitura: mesmo que algum comando tentasse gravar, o SQLite recusaria.
# As 11 execuções do experimento ficam intactas.
repo.engine = create_engine(f"sqlite:///file:{BANCO.as_posix()}?mode=ro&uri=true")

# Serviço que calcula as métricas e o teste de Diebold-Mariano.
av = AvaliacaoService()
# Série mensal importada da ANAC: dá o valor real do mês anterior a cada mês de teste.
serie = repo.carregar_serie()
# As 11 execuções completas (previsões e métricas), em ordem de número. Como funciona:
# listar_execucoes() devolve da mais recente para a mais antiga e sem as previsões; por isso a
# lista é ordenada pelo número e cada execução é buscada inteira com buscar_execucao().
execucoes = [repo.buscar_execucao(e.id) for e in sorted(repo.listar_execucoes(), key=lambda e: e.id)]
# Nomes curtos dos meses e nomes dos modelos, usados nas tabelas exibidas.
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
NOME = {"naive": "Naïve sazonal", "persistencia": "Persistência",
        "random_forest": "Random Forest", "lstm": "LSTM"}
# Dicionário que junta os resultados ao longo do notebook; no fim, vira o persistencia.json.
RES = {"banco": BANCO.name, "execucoes_lidas": [e.id for e in execucoes]}
# Mostra o banco lido e os números das execuções, para conferir.
print("base:", BANCO, "| execuções lidas:", RES["execucoes_lidas"])

base: C:\tcc-previsao-demanda\dados\experimentos.db | execuções lidas: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


## 1. Previsões da persistência

O conjunto de teste é tirado das próprias execuções gravadas, e o notebook confere que as 11 usam
os mesmos 12 meses. O valor do mês anterior vem da série importada da ANAC.

In [2]:
# Valor real de cada mês da série, por (ano, mês), e o conjunto dos meses da pandemia.
valor = {(r.ano, r.mes): r.desembarques for r in serie.registros}
pandemia = {(r.ano, r.mes) for r in serie.registros if r.pandemia}

# Retorna o (ano, mês) do mês anterior. Como funciona: volta um mês; em janeiro, volta para
# dezembro do ano anterior.
def mes_anterior(ano, mes):
    return (ano, mes - 1) if mes > 1 else (ano - 1, 12)

# Meses de teste, tirados da execução do naïve sazonal. Confere que são 12 e que as 11
# execuções usam os mesmos meses.
naive = next(e for e in execucoes if e.tipo_modelo == T.NAIVE)
meses_teste = [(p.ano, p.mes) for p in naive.previsoes]
assert len(meses_teste) == 12
assert all([(p.ano, p.mes) for p in e.previsoes] == meses_teste for e in execucoes), "testes diferentes entre execuções"

# Previsões da persistência. Como funciona: para cada mês de teste, o previsto é o valor real
# do mês anterior; antes, confere que esse mês existe e não é da pandemia e que o valor real
# gravado na execução é o mesmo da série.
persistencia = []
for p in naive.previsoes:
    anterior = mes_anterior(p.ano, p.mes)
    assert anterior in valor and anterior not in pandemia, f"{p.rotulo}: mês anterior indisponível"
    assert p.valor_real == valor[(p.ano, p.mes)], f"{p.rotulo}: valor real diferente da série"
    persistencia.append(Previsao(ano=p.ano, mes=p.mes, valor_previsto=valor[anterior],
                                 valor_real=p.valor_real, alta_temporada=p.alta_temporada))

# Retorna o erro percentual de uma previsão, na mesma convenção da tabela de previsões mensais
# do TCC (Seção 5.3): (real - previsto) / real; positivo = subestimação.
def erro_pct(p):
    return 100 * (p.valor_real - p.valor_previsto) / p.valor_real

# Execução principal de cada modelo: a da semente 42 (config.SEED).
PRINCIPAL = {t.value: next(e for e in execucoes if e.tipo_modelo == t and e.semente_aleatoria == config.SEED)
             for t in (T.NAIVE, T.RANDOM_FOREST, T.LSTM)}
# Tabela mês a mês: real, previsto e erro da persistência, ao lado do erro de cada modelo.
linhas = []
for i, p in enumerate(persistencia):
    linha = {"mes": f"{MESES[p.mes - 1]}/{str(p.ano)[2:]}", "alta": p.alta_temporada,
             "real": p.valor_real, "persistencia_previsto": p.valor_previsto,
             "persistencia_erro_pct": round(erro_pct(p), 1)}
    for modelo, ex in PRINCIPAL.items():
        linha[f"{modelo}_erro_pct"] = round(erro_pct(ex.previsoes[i]), 1)
    linhas.append(linha)
RES["previsoes"] = linhas
# Conta em quantos meses a persistência previu abaixo do real (subestimou).
RES["meses_subestimados"] = sum(p.valor_real > p.valor_previsto for p in persistencia)
print("meses subestimados pela persistência:", RES["meses_subestimados"], "de 12")
# Mostra a tabela.
pd.DataFrame(linhas)

meses subestimados pela persistência: 6 de 12


,mes,alta,real,persistencia_previsto,persistencia_erro_pct,naive_erro_pct,random_forest_erro_pct,lstm_erro_pct
0,jan/25,True,105033.0,82734.0,21.2,10.1,15.5,9.1
1,fev/25,False,89051.0,105033.0,-17.9,4.9,-0.7,0.4
2,mar/25,False,85772.0,89051.0,-3.8,4.2,-1.7,12.8
3,abr/25,False,82034.0,85772.0,-4.6,11.7,-10.3,12.2
4,mai/25,False,83427.0,82034.0,1.7,14.3,0.4,13.1
5,jun/25,False,82391.0,83427.0,-1.3,15.0,1.3,4.5
6,jul/25,True,88891.0,82391.0,7.3,-0.9,4.2,3.2
7,ago/25,False,96301.0,88891.0,7.7,9.6,13.1,7.8
8,set/25,False,89102.0,96301.0,-8.1,9.4,3.7,7.6
9,out/25,False,101975.0,89102.0,12.6,15.1,14.0,15.6


## 2. Métricas (coluna nova da tabela de métricas de erro no teste, Seção 5.3)

Calculadas pela mesma função que o sistema usa em todas as execuções. Antes, o notebook confere
que essa função reproduz as métricas gravadas das 11 execuções.

In [3]:
# Confere que a função de métricas do sistema reproduz as métricas gravadas das 11 execuções.
# Como funciona: recalcula MAE, RMSE e MAPE de cada execução a partir das previsões e compara
# com o que está no banco; interrompe se algum valor não conferir.
for e in execucoes:
    gravadas = {(m.escopo_avaliacao, m.tipo_metrica): m.valor_metrica for m in e.metricas}
    recalculadas = {(m.escopo_avaliacao, m.tipo_metrica): m.valor_metrica for m in av.avaliar(e.previsoes)}
    assert all(abs(gravadas[k] - v) < 1e-6 for k, v in recalculadas.items()), f"execução {e.id} não confere"
print("métricas gravadas das 11 execuções reproduzidas pela mesma função: ok")

# Retorna as métricas de uma lista de previsões num dicionário ("global_MAE",
# "alta_temporada_MAPE" etc.). Como funciona: chama av.avaliar(), a mesma função do sistema, e
# dá nome a cada valor pelo escopo e pela métrica.
def metricas(previsoes):
    return {f"{m.escopo_avaliacao.value}_{m.tipo_metrica.value}": m.valor_metrica for m in av.avaliar(previsoes)}

# Métricas da persistência e das execuções principais dos três modelos, mostradas numa tabela.
RES["metricas"] = {"persistencia": metricas(persistencia),
                   **{modelo: metricas(ex.previsoes) for modelo, ex in PRINCIPAL.items()}}
pd.DataFrame(RES["metricas"]).T.rename(index=NOME).round(2)

métricas gravadas das 11 execuções reproduzidas pela mesma função: ok


,global_MAE,global_RMSE,global_MAPE,alta_temporada_MAE,alta_temporada_RMSE,alta_temporada_MAPE
Persistência,7343.42,9688.36,7.69,11439.67,13783.64,11.24
Naïve sazonal,10257.75,11766.06,10.87,11744.00,15079.79,11.12
Random Forest,6671.61,8604.62,6.85,9884.13,11141.07,9.58
LSTM,8445.87,9460.92,9.01,8553.87,9564.16,8.25


## 3. Comparação com as cinco sementes (linha nova da tabela de MAPE nas cinco sementes, Seção 5.4)

A persistência é determinística, como o naïve sazonal: tem um único valor por escopo. Para cada
modelo, conta-se em quantas sementes o MAPE ficou abaixo dela.

In [4]:
# MAPE da persistência e do naïve sazonal, no ano todo (global) e na alta temporada.
mape_p = {esc: RES["metricas"]["persistencia"][f"{esc}_MAPE"] for esc in ("global", "alta_temporada")}
mape_n = {esc: RES["metricas"]["naive"][f"{esc}_MAPE"] for esc in ("global", "alta_temporada")}
# Resumo das cinco sementes do Random Forest e da LSTM, em cada escopo. Como funciona: junta o
# MAPE das cinco execuções do modelo e calcula média, desvio-padrão, mínimo e máximo, as
# sementes em que o MAPE ficou abaixo do da persistência e a redução percentual do erro em
# relação à persistência e ao naïve sazonal, na média e na semente 42.
resumo = []
for tipo in (T.RANDOM_FOREST, T.LSTM):
    do_tipo = [e for e in execucoes if e.tipo_modelo == tipo]
    for esc, enum in (("global", E.GLOBAL), ("alta_temporada", E.ALTA_TEMPORADA)):
        v = pd.Series([e.metrica(enum, M.MAPE) for e in do_tipo], index=[e.semente_aleatoria for e in do_tipo])
        resumo.append({
            "modelo": tipo.value, "escopo": esc, "media": v.mean(), "dp": v.std(), "min": v.min(), "max": v.max(),
            "sementes_abaixo_da_persistencia": [int(s) for s in v.index[v < mape_p[esc]]],
            "reducao_media_vs_persistencia_pct": 100 * (mape_p[esc] - v.mean()) / mape_p[esc],
            "reducao_media_vs_naive_pct": 100 * (mape_n[esc] - v.mean()) / mape_n[esc],
            "reducao_semente42_vs_persistencia_pct": 100 * (mape_p[esc] - v[config.SEED]) / mape_p[esc],
            "reducao_semente42_vs_naive_pct": 100 * (mape_n[esc] - v[config.SEED]) / mape_n[esc],
        })
RES["persistencia_mape"] = mape_p
RES["sementes"] = resumo
# Mostra o resumo.
pd.DataFrame(resumo).round(2)

,modelo,escopo,media,dp,min,max,sementes_abaixo_da_persistencia,reducao_media_vs_persistencia_pct,reducao_media_vs_naive_pct,reducao_semente42_vs_persistencia_pct,reducao_semente42_vs_naive_pct
0,random_forest,global,6.55,0.33,6.13,6.88,"[42, 43, 44, 45, 46]",14.86,39.77,10.96,37.02
1,random_forest,alta_temporada,7.67,1.30,6.26,9.58,"[42, 43, 44, 45, 46]",31.80,31.03,14.77,13.81
2,lstm,global,8.32,0.50,7.60,9.01,[45],-8.21,23.45,-17.24,17.06
3,lstm,alta_temporada,9.20,1.42,7.84,11.49,"[42, 43, 44, 45]",18.17,17.25,26.62,25.79


## 4. Significância estatística: Diebold-Mariano contra a persistência

Mesma função e mesmas convenções da Seção 5.5 do TCC: hipótese nula de acurácia igual; estatística
negativa indica que o **primeiro** modelo do par errou menos; perda quadrática (principal) e absoluta
(sensibilidade). No escopo de alta temporada (n = 3) os valores são só descritivos.

In [5]:
# Pares comparados no teste: cada modelo contra a persistência.
pares = [("random_forest", "persistencia"), ("lstm", "persistencia"), ("naive", "persistencia")]
# Previsões de cada modelo (execuções principais, semente 42) e da persistência.
previsoes = {**{m: ex.previsoes for m, ex in PRINCIPAL.items()}, "persistencia": persistencia}
# Diebold-Mariano de cada par, nos dois escopos e nas duas perdas (quadrática e absoluta). Como
# funciona: separa os erros dos dois modelos no escopo, chama a mesma função do sistema e marca
# como significativo só quando o teste é aplicável e o valor-p fica abaixo de 5%.
dm = []
for a, b in pares:
    for esc, filtro in (("global", lambda p: True), ("alta_temporada", lambda p: p.alta_temporada)):
        ea = [p.erro for p in previsoes[a] if filtro(p)]
        eb = [p.erro for p in previsoes[b] if filtro(p)]
        for perda in ("quadratica", "absoluta"):
            r = av.teste_diebold_mariano(ea, eb, perda=perda)
            dm.append({"par": f"{a} x {b}", "escopo": esc, "perda": perda, **r,
                       "significativo_5": bool(r["aplicavel"] and r["valor_p"] is not None and r["valor_p"] < 0.05)})
RES["diebold_mariano"] = dm
# Mostra a tabela do teste.
pd.DataFrame(dm)[["par", "escopo", "perda", "n", "estatistica", "valor_p", "aplicavel", "significativo_5"]].round(3)

,par,escopo,perda,n,estatistica,valor_p,aplicavel,significativo_5
0,random_forest x persistencia,global,quadratica,12,-0.608,0.556,True,False
1,random_forest x persistencia,global,absoluta,12,-0.379,0.712,True,False
2,random_forest x persistencia,alta_temporada,quadratica,3,-0.758,0.528,False,False
3,random_forest x persistencia,alta_temporada,absoluta,3,-0.521,0.654,False,False
4,lstm x persistencia,global,quadratica,12,-0.091,0.929,True,False
5,lstm x persistencia,global,absoluta,12,0.465,0.651,True,False
6,lstm x persistencia,alta_temporada,quadratica,3,-0.608,0.605,False,False
7,lstm x persistencia,alta_temporada,absoluta,3,-0.488,0.674,False,False
8,naive x persistencia,global,quadratica,12,0.687,0.506,True,False
9,naive x persistencia,global,absoluta,12,1.076,0.305,True,False


In [6]:
# Robustez: cada execução do RF e da LSTM (as cinco sementes) contra a persistência, no escopo
# global e nas duas perdas. Como funciona: aplica o Diebold-Mariano execução por execução,
# pulando o naïve, e guarda a estatística e o valor-p de cada uma.
erros_p = [p.erro for p in persistencia]
rob = []
for e in execucoes:
    if e.tipo_modelo == T.NAIVE:
        continue
    for perda in ("quadratica", "absoluta"):
        r = av.teste_diebold_mariano([p.erro for p in e.previsoes], erros_p, perda=perda)
        rob.append({"execucao": e.id, "modelo": e.tipo_modelo.value, "semente": e.semente_aleatoria,
                    "perda": perda, "estatistica": r["estatistica"], "valor_p": r["valor_p"]})
RES["diebold_mariano_por_semente"] = rob
# Menor e maior valor-p de cada modelo e perda, e se algum ficou abaixo de 5%.
faixas = pd.DataFrame(rob).groupby(["modelo", "perda"]).valor_p.agg(["min", "max"]).round(3)
RES["faixa_valor_p_por_semente"] = {f"{m} {p}": list(v) for (m, p), v in faixas.iterrows()}
print("algum valor-p abaixo de 5%?", any(x["valor_p"] < 0.05 for x in rob))
faixas

algum valor-p abaixo de 5%? False


min    max
modelo        perda                   
lstm          absoluta    0.651  0.996
              quadratica  0.702  0.945
random_forest absoluta    0.446  0.712
              quadratica  0.325  0.556

## 5. Gravação dos resultados

In [7]:
# Retorna uma cópia dos resultados que o json consegue gravar. Como funciona: percorre
# dicionários e listas, passa as chaves para texto e troca os números e os valores lógicos do
# numpy pelos tipos comuns do Python (float, int e bool).
def limpar(o):
    if isinstance(o, dict): return {str(k): limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [limpar(v) for v in o]
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    return o
# Grava os resultados em persistencia.json, na pasta de saída, e mostra que o banco continua
# com as 11 execuções (nada foi gravado nele).
(SAIDA / "persistencia.json").write_text(json.dumps(limpar(RES), ensure_ascii=False, indent=1), encoding="utf-8")
print("gravado:", SAIDA / "persistencia.json")
print("execuções na base (inalterada):", len(repo.listar_execucoes()))

gravado: C:\tcc-previsao-demanda\docs\resultados\persistencia.json
execuções na base (inalterada): 11
